# 2. Random sampling and warm start

This standalone notebook teaches predictive sampling for a differential-drive robot.
We control forward speed **v** and turning speed **omega** to reach a target position.
We use the three circular obstacles from `predictive_sampling.py`. The final heading is unconstrained.

At every control step:
1. Sample candidate future control plans.
2. Simulate the robot under each plan.
3. Select the plan with the lowest cost.
4. Apply only its first control, observe the new state, and repeat.

This is receding-horizon control (MPC). We select the best sample; we do not average samples or use gradients.

**Requirements:** Python, NumPy, Matplotlib, and a Jupyter notebook environment.
If needed, run `%pip install numpy matplotlib` in a separate cell. Then run this notebook from top to bottom.
All four notebooks use the same model, cost, sample count, horizon, and random seed.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Re-running this cell resets the random sequence.
rng = np.random.default_rng(7)
dt = 0.1                         # Control interval [s]
horizon = 20                     # Future controls: 2 seconds
num_samples = 512                # Candidate plans per control step
max_steps = 200                  # Maximum simulation length
goal_tolerance = 0.15            # Stop within this distance [m]
start = np.array([0.0, 0.0, 0.0]) # x [m], y [m], heading [rad]
goal = np.array([4.0, 2.0])       # Target position [m]
control_low = np.array([-1.0, -1.5])  # Minimum v [m/s], omega [rad/s]
control_high = np.array([1.0, 1.5])  # Maximum v [m/s], omega [rad/s]

# Each row contains obstacle center x [m], center y [m], and radius [m].
obstacles = np.array([
    [1.20, 0.35, 0.28],
    [2.15, 1.05, 0.35],
    [3.05, 1.35, 0.30],
])
robot_radius = 0.15  # Robot footprint [m]
safety_margin = 0.15  # Desired extra clearance [m]
obstacle_weight = 250.0
collision_weight = 10000.0


## 1. Predict the robot motion

We use forward Euler integration of the unicycle kinematics:

$$x_{t+1}=x_t+\Delta t\,v_t\cos\theta_t,\qquad
 y_{t+1}=y_t+\Delta t\,v_t\sin\theta_t,\qquad
 \theta_{t+1}=\theta_t+\Delta t\,\omega_t.$$

The same function handles one state with shape `(3,)` or a batch with shape `(num_samples, 3)`.
Batching candidates keeps the notebook fast while retaining an explicit loop over prediction time.


In [ ]:
def robot_step(state, control):
    """Advance one state, or a batch of states, by one time step."""
    next_state = np.empty_like(state)
    next_state[..., 0] = state[..., 0] + dt * control[..., 0] * np.cos(state[..., 2])
    next_state[..., 1] = state[..., 1] + dt * control[..., 0] * np.sin(state[..., 2])
    next_state[..., 2] = state[..., 2] + dt * control[..., 1]
    return next_state


def rollout(initial_state, controls):
    """Simulate controls shaped (candidates, horizon, 2)."""
    states = np.empty((len(controls), horizon + 1, 3))
    states[:, 0] = initial_state
    for t in range(horizon):
        states[:, t + 1] = robot_step(states[:, t], controls[:, t])
    return states


def trajectory_cost(states, controls):
    """Score goal progress, control effort, and obstacle proximity."""
    squared_distance = np.sum((states[:, 1:, :2] - goal) ** 2, axis=2)
    effort = controls[:, :, 0] ** 2 + 0.1 * controls[:, :, 1] ** 2
    # Surface-to-surface clearance includes the robot's circular footprint.
    obstacle_cost = np.zeros(len(controls))
    for obstacle_x, obstacle_y, radius in obstacles:
        distance = np.linalg.norm(
            states[:, 1:, :2] - np.array([obstacle_x, obstacle_y]), axis=2)
        clearance = distance - radius - robot_radius
        margin_violation = np.maximum(safety_margin - clearance, 0.0)
        obstacle_cost += np.sum(obstacle_weight * margin_violation ** 2
                                + collision_weight * (clearance <= 0.0), axis=1)
    return (dt * np.sum(squared_distance + 0.02 * effort, axis=1)
            + 10.0 * squared_distance[:, -1] + obstacle_cost)


The cost adds squared position errors along the predicted path, a small control-effort penalty,
and a terminal position penalty. Lower is better. The terminal term encourages progress beyond the next step.
This finite random search is approximate: it does not guarantee an optimal plan or convergence.


For each obstacle, subtract its radius and the robot radius from the center distance.
Negative clearance means collision. Entering the safety margin adds a quadratic penalty;
collision adds a much larger penalty. The obstacle weights and per-step penalty match
`predictive_sampling.py`; the simple goal and effort costs remain those of this tutorial.
These are soft costs evaluated at predicted states, not hard collision constraints.


## 2. Represent a control plan

Sample one independent `(v, omega)` pair per prediction step: **40 scalar parameters**. Each control is held constant over its time step. Adjacent controls can differ sharply.


## 3. Sample and select

At the first step, draw uniform random plans. Later, shift the previous best plan forward by **one control step**, hold its last value at the end, and add uniform random perturbations. Keep the shifted plan itself as candidate zero. Clip perturbations to the velocity limits. This notebook uses local sampling after initialization; the perturbation size controls how far the search can move.


In [ ]:
def sample_plans(previous_plan):
    """Draw control sequences for the next optimization."""
    if previous_plan is None:
        return rng.uniform(control_low, control_high, (num_samples, horizon, 2))

    # Discard the applied control and repeat the last control at the tail.
    nominal = np.concatenate((previous_plan[1:], previous_plan[-1:]), axis=0)
    noise = rng.uniform(-1.0, 1.0, (num_samples, horizon, 2))
    candidates = np.clip(nominal + noise * np.array([0.2, 0.4]),
                         control_low, control_high)
    candidates[0] = nominal  # Always evaluate the unperturbed warm start.
    return candidates


In [ ]:
def predictive_sampling(state, previous_plan=None):
    """Return the best plan and predictions for inspecting this search."""
    plans = sample_plans(previous_plan)
    controls = plans
    predictions = rollout(state, controls)
    costs = trajectory_cost(predictions, controls)
    best = np.argmin(costs)
    return plans[best].copy(), controls[best].copy(), predictions, best


## 4. Run the closed loop

Only the first control of the winning plan is executed. The remaining controls are predictions, not a committed path. The stopping condition depends only on position. We explicitly apply zero velocity once inside the tolerance.


In [ ]:
# Reset here so this entire experiment can be repeated independently.
rng = np.random.default_rng(7)
state = start.copy()
previous_plan = None
state_history = [state.copy()]
control_history = []

for step in range(max_steps):
    if np.linalg.norm(state[:2] - goal) <= goal_tolerance:
        break
    best_plan, best_controls, predictions, best = predictive_sampling(state, previous_plan)
    if step == 0:
        first_predictions = predictions.copy()
        first_best = best
        first_plan = best_plan.copy()
        first_controls = best_controls.copy()
    control = best_controls[0]
    state = robot_step(state, control)
    state_history.append(state.copy())
    control_history.append(control.copy())
    previous_plan = best_plan

reached = np.linalg.norm(state[:2] - goal) <= goal_tolerance
if reached:
    control_history.append(np.zeros(2))
    state_history.append(robot_step(state, np.zeros(2)))
state_history = np.asarray(state_history)
control_history = np.asarray(control_history)
print(f"Goal reached: {reached}")
print(f"Final position error: {np.linalg.norm(state[:2] - goal):.3f} m")
print(f"Executed control steps (including stop if reached): {len(control_history)}")

# Check clearance along each executed straight Euler step, not just its endpoints.
segment_start = state_history[:-1, :2]
segment_delta = np.diff(state_history[:, :2], axis=0)
segment_length_squared = np.sum(segment_delta ** 2, axis=1)
minimum_clearance = np.inf
for obstacle_x, obstacle_y, radius in obstacles:
    center = np.array([obstacle_x, obstacle_y])
    projection = np.sum((center - segment_start) * segment_delta, axis=1)
    fraction = np.clip(projection / np.maximum(segment_length_squared, 1e-12), 0, 1)
    closest_point = segment_start + fraction[:, None] * segment_delta
    clearance = np.linalg.norm(closest_point - center, axis=1) - radius - robot_radius
    minimum_clearance = min(minimum_clearance, clearance.min())
print(f"Minimum executed clearance: {minimum_clearance:.3f} m")
print(f"Collision-free executed path: {minimum_clearance > 0}")


## 5. Inspect the predictions and executed motion

The left plot shows candidates from the **first** search, including its winner. The right plot shows the actual path obtained by replanning at every step. They need not match.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for prediction in first_predictions[:40]:
    axes[0].plot(prediction[:, 0], prediction[:, 1], color="gray", alpha=0.25)
axes[0].plot(first_predictions[first_best, :, 0], first_predictions[first_best, :, 1],
             color="tab:orange", linewidth=2, label="Best prediction")
axes[1].plot(state_history[:, 0], state_history[:, 1], label="Executed path")
for ax, title in zip(axes, ["First search: 40 samples and winner", "Closed-loop motion"]):
    ax.scatter(*start[:2], label="Start", color="black")
    ax.scatter(*goal, marker="*", s=150, label="Goal", color="tab:green")
    for index, (obstacle_x, obstacle_y, radius) in enumerate(obstacles):
        ax.add_patch(plt.Circle((obstacle_x, obstacle_y), radius,
                               color="tab:red", alpha=0.6,
                               label="Obstacle" if index == 0 else None))
        ax.add_patch(plt.Circle((obstacle_x, obstacle_y),
                               radius + robot_radius + safety_margin,
                               fill=False, linestyle="--", color="tab:red",
                               label="Robot center safety boundary" if index == 0 else None))
    ax.set(xlabel="x [m]", ylabel="y [m]", title=title)
    ax.axis("equal")
    ax.grid(True)
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
control_time = np.arange(len(control_history)) * dt
for channel, label in enumerate(["v [m/s]", "omega [rad/s]"]):
    axes[channel].step(control_time, control_history[:, channel], where="post")
    axes[channel].set_ylabel(label)
    axes[channel].grid(True)
axes[0].set_title("Executed controls after replanning")
axes[1].set_xlabel("Time [s]")
plt.tight_layout()
plt.show()


## 6. Small experiments

- Reduce `num_samples` from 512 to 64. How does the executed path change?
- Change the seed and rerun. A single run is not a reliable performance comparison.
- Increase the horizon. More look-ahead also makes the search harder.
- Change the perturbation limits `[0.2, 0.4]`. Small noise refines the old plan; larger noise explores further.
- Warm start preserves useful plans, but can also limit exploration.

For a fair comparison, use the same settings and several seeds in all four notebooks.
